# **PROYECTO INTEGRAL: SISTEMA RAG Y AGENTE IA CON MISTRAL AI Y LANGCHAIN**
### **Entorno Optimizado para Google Colab**

Este cuaderno interactivo contiene la implementación completa y secuencial de las **8 fases** de ingeniería de software para construir un pipeline avanzado de Recuperación Aumentada por Generación (RAG) y un Agente Autónomo sobre datos transaccionales de ventas (`sales_data_sample.csv`).

---

## **FASE 1: COMPRENSIÓN TEÓRICA Y ARQUITECTURA**

### **Conceptos Clave**
- **Corpus:** Conjunto estructurado de datos lingüísticos (nuestro CSV convertido en narrativas).
- **RAG (Retrieval-Augmented Generation):** Arquitectura que optimiza la salida de un LLM consultando una fuente de conocimiento externa antes de generar la respuesta.
- **Embeddings:** Vectores numéricos que capturan el significado semántico profundo de un texto.
- **Vector Store (FAISS):** Base de datos indexada en memoria optimizada para búsquedas matemáticas de similitud espacial.
- **Agente IA:** Sistema autónomo que usa el LLM como motor de razonamiento para decidir qué herramientas externas ejecutar de forma iterativa.

## **FASE 2: CARGA Y ANÁLISIS DEL CSV**
Ejecuta la siguiente celda para realizar un diagnóstico de la estructura física del dataset y evaluar la estrategia de fragmentación (*chunking*).

In [45]:
import pandas as pd

def analizar_estructura_corpus(path_csv):
    print("==================================================================")
    print("      DIAGNÓSTICO ARQUITECTÓNICO DEL DATASET (FASE 2)")
    print("==================================================================\n")

    # Carga inicial diagnóstica
    df = pd.read_csv(path_csv, encoding='unicode_escape')
    filas, columnas = df.shape
    print(f"[ESTRUCTURA] El dataset cuenta con {filas} filas y {columnas} columnas.\n")

    # Mapeo de nulos y tipos
    info_columnas = pd.DataFrame({
        'Tipo_Dato_Nativo': df.dtypes,
        'Valores_No_Nulos': df.count(),
        'Valores_Nulos': df.isnull().sum(),
        '%_Nulos': (df.isnull().sum() / filas) * 100
    })
    print("--- 1. MAPEO DE COLUMNAS ---")
    print(info_columnas.head(10))

    # Análisis de longitud de strings (Vital para Chunking)
    longitudes_filas = df.astype(str).apply(lambda x: ', '.join(x), axis=1).str.len()
    print(f"\n--- 2. MÉTRICAS TEXTUALES ---")
    print(f"Longitud promedio de una fila en caracteres: {longitudes_filas.mean():.2f}")
    print(f"Estimación aproximada de tokens promedio por fila: {longitudes_filas.mean() / 4:.1f} tokens.")
    print("\n[CONCLUSIÓN] Como cada fila consume < 100 tokens, la fila entera será nuestra unidad mínima (Chunk).")

# Nota: Asegúrate de subir 'sales_data_sample.csv' en Colab antes de correrlo
try:
    analizar_estructura_corpus('sales_data_sample.csv')
except FileNotFoundError:
    print("[ALERTA] Por favor, sube el archivo 'sales_data_sample.csv' al panel izquierdo de Colab.")

      DIAGNÓSTICO ARQUITECTÓNICO DEL DATASET (FASE 2)

[ESTRUCTURA] El dataset cuenta con 2823 filas y 25 columnas.

--- 1. MAPEO DE COLUMNAS ---
                Tipo_Dato_Nativo  Valores_No_Nulos  Valores_Nulos  %_Nulos
ORDERNUMBER                int64              2823              0      0.0
QUANTITYORDERED            int64              2823              0      0.0
PRICEEACH                float64              2823              0      0.0
ORDERLINENUMBER            int64              2823              0      0.0
SALES                    float64              2823              0      0.0
ORDERDATE                 object              2823              0      0.0
STATUS                    object              2823              0      0.0
QTR_ID                     int64              2823              0      0.0
MONTH_ID                   int64              2823              0      0.0
YEAR_ID                    int64              2823              0      0.0

--- 2. MÉTRICAS TEXTUALES --

## **FASE 3: NORMALIZACIÓN Y LIMPIEZA**
En esta celda corregimos espacios, estandarizamos mayúsculas, mitigamos nulos y transformamos los tipos de datos temporales para evitar quiebres lógicos en las fases NLP.

In [46]:
import pandas as pd
import numpy as np

def normalizar_dataset(path_entrada):
    print("[PROCESO] Iniciando normalización del dataset...")
    df = pd.read_csv(path_entrada, encoding='unicode_escape')

    # Limpieza de encabezados
    df.columns = df.columns.str.strip().str.upper()

    # Conversión cronológica estricta
    if 'ORDERDATE' in df.columns:
        df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], errors='coerce')

    # Remoción de espacios ocultos en celdas de texto
    columnas_texto = df.select_dtypes(include=['object']).columns
    for col in columnas_texto:
        df[col] = df[col].astype(str).str.strip()

    # Mitigación semántica de valores nulos críticos
    columnas_nulos = ['ADDRESSLINE2', 'STATE', 'POSTALCODE', 'TERRITORY']
    for col in columnas_nulos:
        if col in df.columns:
            df[col] = df[col].replace(['nan', 'NaN', '', None], 'No Registrado').fillna('No Registrado')

    # Homogeneización de categorías
    columnas_cat = ['STATUS', 'PRODUCTLINE', 'DEALSIZE', 'COUNTRY']
    for col in columnas_cat:
        if col in df.columns:
            df[col] = df[col].str.upper()

    print(f"[ÉXITO] Limpieza concluida. Nulos remanentes en el sistema: {df.isnull().sum().sum()}")
    return df

try:
    df_limpio = normalizar_dataset('sales_data_sample.csv')
    df_limpio.to_csv('ventas_normalizado.csv', index=False)
    print("[SISTEMA] Archivo depurado guardado como 'ventas_normalizado.csv'.")
except Exception as e:
    print(f"[ERROR] No se pudo procesar la limpieza: {e}")

[PROCESO] Iniciando normalización del dataset...
[ÉXITO] Limpieza concluida. Nulos remanentes en el sistema: 0
[SISTEMA] Archivo depurado guardado como 'ventas_normalizado.csv'.


## **FASE 4 Y 5: EVALUACIÓN DEL MODELO MISTRAL**
Elegimos **Mistral Large 3** (`mistral-large-latest`) consumido vía API remota. Esto nos otorga una enorme capacidad de razonamiento lógico, llamadas nativas a funciones (*Function Calling*) y una ventana de **256K tokens**, liberando al hardware de Colab de cargas insostenibles.

## **FASE 6: CONFIGURACIÓN DEL ENTORNO DE COLAB**
Instala las dependencias necesarias de LangChain, FAISS y Mistral. Luego, carga de forma segura tu credencial desde el panel de **Secretos (ícono de llave)**.

In [47]:
# Instalación silenciosa de dependencias del ecosistema de IA
!pip install -q mistralai langchain langchain-community langchain-mistralai faiss-cpu sentence-transformers
print("[ENTORNO] Dependencias instaladas de manera correcta.")

[ENTORNO] Dependencias instaladas de manera correcta.


In [48]:
import os
from google.colab import userdata
from langchain_mistralai import ChatMistralAI

def inicializar_api_mistral():
    try:
        # Extrae la API Key configurada previamente en los Secretos de tu entorno Colab
        api_key = userdata.get('MISTRAL_API_KEY')
        os.environ["MISTRAL_API_KEY"] = api_key

        # Instanciación determinista del motor inteligente (Temperatura = 0)
        llm = ChatMistralAI(model="mistral-large-latest", temperature=0.0)

        # Prueba sintáctica rápida (Ping de control)
        test_response = llm.invoke("Confirma conexión respondiendo únicamente: CONEXIÓN EXITOSA")
        print(f"[API MISTRAL] {test_response.content.strip()}")
        return llm
    except Exception as e:
        print("[ERROR CRÍTICO] Configura tu 'MISTRAL_API_KEY' en la pestaña de Secretos (llave lateral izquierda).")
        return None

llm_configurado = inicializar_api_mistral()

[API MISTRAL] CONEXIÓN EXITOSA


## **FASE 7: CONSTRUCCIÓN DEL SISTEMA RAG**
Esta celda ejecuta la **Textualización Semántica** de las filas transaccionales, genera los embeddings densos mediante *Sentence Transformers*, construye el índice vectorial indexado con *FAISS* y ensambla la cadena RAG definitiva de LangChain.

In [ ]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import pandas as pd

def ensamblar_sistema_rag(path_csv_normalizado, llm_motor):
    df = pd.read_csv(path_csv_normalizado)
    documentos_rag = []

    print("[RAG] Transformando filas rígidas en narrativas conceptuales de alta semántica...")
    for idx, fila in df.iterrows():
        narrativa = (
            f"Registro de Venta - Pedido Nro: {fila.get('ORDERNUMBER', 'N/R')}. "
            f"Fecha de operación: {str(fila.get('ORDERDATE', 'N/R')).split()[0]}. "
            f"Estado actual: {fila.get('STATUS', 'N/R')}. "
            f"Línea de Producto: '{fila.get('PRODUCTLINE', 'N/R')}'. "
            f"Cantidad de unidades adquiridas: {fila.get('QUANTITYORDERED', 0)}. "
            f"Monto total facturado en esta transacción: {fila.get('SALES', 0)} USD. "
            f"Cliente: {fila.get('CUSTOMERNAME', 'N/R')}, ubicado en País: {fila.get('COUNTRY', 'N/R')}."
        )
        metadatos = {"country": str(fila.get('COUNTRY')).upper(), "order_number": str(fila.get('ORDERNUMBER'))}
        documentos_rag.append(Document(page_content=narrativa, metadata=metadatos))

    # Generación de vectores en local (CPU de Colab)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_db = FAISS.from_documents(documentos_rag, embeddings)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    # Prompt de Blindaje Corporativo contra alucinaciones
    # A diferencia del 'create_pandas_dataframe_agent' que requiere un 'prefix'
    # estático describiendo las columnas, aquí el contexto del dataset se genera
    # de forma DINÁMICA. El RAG (FAISS) busca las filas relevantes y las inyecta
    # automáticamente en la variable {context} abajo, optimizando el uso de tokens.
    system_prompt = (
        "Eres un Analista Senior de Datos Financieros.\n"
        "Responde las preguntas de negocio basándote estricta y exclusivamente en el contexto verídico provisto abajo.\n"
        "Si el contexto no tiene los datos para responder o calcular el dato preciso, di textualmente: "
        "'Lo siento, la información disponible en el corpus analizado no contiene los datos específicos para responder esa pregunta.'\n\n"
        "CONTEXTO FACTUAL:\n{context}" # <-- El dataset entra aquí dinamizado por el RAG
    )
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
    ])

    # Auxiliar para formatear los documentos recuperados de FAISS pasándolos a texto plano
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # CONSTUCCIÓN DE LA CADENA CON LCEL (Substituye create_retrieval_chain)
    # Define un flujo limpio: recupera contexto, arma el prompt, ejecuta Mistral y parsea la respuesta.
    rag_chain_moderna = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt_template
        | llm_motor
        | StrOutputParser()
    )

    # Creamos una interfaz compatible para que devuelva el formato dict tradicional {"answer": ...}
    class RagWrapper:
        def __init__(self, chain):
            self.chain = chain
        def invoke(self, inputs):
            res = self.chain.invoke(inputs["input"])
            return {"answer": res}

    return RagWrapper(rag_chain_moderna)

# =====================================================================
# EJECUCIÓN CON ASIGNACIÓN GLOBAL EN EL ENTORNO DE COLAB
# =====================================================================
if 'llm_configurado' in locals() and llm_configurado is not None:
    try:
        # Asignamos el pipeline directamente a la variable que usará la Fase 8
        pipeline_rag = ensamblar_sistema_rag('ventas_normalizado.csv', llm_configurado)
        print("[ÉXITO] 'pipeline_rag' guardado con éxito mediante sintaxis LCEL moderna.")

        # Prueba de control interna
        consulta = "¿Qué pedidos se realizaron para la línea de productos 'Motorcycles' en 'France'?"
        resultado = pipeline_rag.invoke({"input": consulta})
        print(f"\n[MISTRAL RAG TEST]:\n{resultado['answer']}")

    except FileNotFoundError:
        print("[ERROR] No se encuentra 'ventas_normalizado.csv'. Recuerda ejecutar primero la Fase 3.")
else:
    print("[ERROR CRÍTICO] El modelo 'llm_configurado' de la Fase 6 no está en memoria.")

[RAG] Transformando filas rígidas en narrativas conceptuales de alta semántica...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ÉXITO] 'pipeline_rag' guardado con éxito mediante sintaxis LCEL moderna.


## **FASE 8: AGENTE IA (AUTONOMÍA COMPLETA)**
Elevamos el sistema registrando herramientas funcionales. El modelo **Mistral Large 3** implementará un ciclo de razonamiento lógico interactivo (Bucle ReAct), auto-evaluando si requiere buscar información en el RAG o delegar operaciones matemáticas exactas al intérprete nativo de Python.

In [53]:
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, ToolMessage
import pandas as pd
import time  # <--- Importamos la librería de tiempo

# --- DEFINICIÓN DE CAPACIDADES EXTERNAS (HERRAMIENTAS) ---
def tool_buscador_rag(query):
    res = pipeline_rag.invoke({"input": query})
    return res["answer"]

def tool_calculadora_python(expresion_matematica):
    try:
        res = eval(str(expresion_matematica), {"__builtins__": None}, {})
        return f"Resultado analítico exacto de Python: {res}"
    except Exception as e:
        return f"No se pudo calcular la expresión. Detalles: {e}"

# El agente de pandas usa 'allow_dangerous_code=True' porque el LLM genera
# y ejecuta código libre sobre todo el DataFrame.
# En esta arquitectura, para mitigar riesgos de seguridad, acotamos el entorno:
# El agente NO toca el DataFrame; solo tiene acceso a estas dos herramientas
# estrictamente supervisadas y controladas por funciones locales.
herramientas_agente = [
    Tool(
        name="Buscador_Ventas_RAG",
        func=tool_buscador_rag,
        description="Útil para buscar registros de transacciones, clientes, fechas y líneas de productos en el CSV de ventas. Recibe una consulta en texto largo."
    ),
    Tool(
        name="Calculadora_Matematica_Python",
        func=tool_calculadora_python,
        description="Útil para realizar sumas numéricas, multiplicaciones y cálculos financieros exactos. Recibe una cadena con números y operadores (ej: '2871.0 + 3500.0')."
    )
]

def ejecutar_agente_nativo_mistral(pregunta, llm, tools):
    print("[SISTEMA] Inicializando Agente Autónomo con Function Calling Nativo...")
    dict_herramientas = {t.name: t.func for t in tools}
    llm_con_herramientas = llm.bind_tools(tools)

    # Este es el equivalente al 'prefix'. Al haber delegado la búsqueda de
    # datos al RAG (Fase 7), el prompt de comportamiento del agente queda limpio
    # y enfocado puramente en el rol de auditoría y el uso lógico de herramientas.

    mensajes = [
        HumanMessage(content=(
            "Eres un Agente Corporativo Autónomo de Auditoría Financiera.\n"
            "Tu misión es resolver la consulta del cliente utilizando las herramientas a tu disposición si es necesario.\n"
            "Sé preciso, profesional y realiza todos los cálculos aritméticos pertinentes de forma explícita.\n\n"
            f"PREGUNTA DEL CLIENTE: {pregunta}"
        ))
    ]

    for paso in range(5):
        # -----------------------------------------------------------------
        # PROTECCIÓN DE CUOTA: Pausa de 2 segundos para evitar el Error 429
        # -----------------------------------------------------------------
        time.sleep(2)

        print(f"\n[PENSAMIENTO] El agente analiza el contexto actual (Paso {paso + 1})...")
        respuesta = llm_con_herramientas.invoke(mensajes)
        mensajes.append(respuesta)

        if hasattr(respuesta, 'tool_calls') and respuesta.tool_calls:
            for tool_call in respuesta.tool_calls:
                nombre_tool = tool_call['name']
                argumentos = tool_call['args']
                id_tool = tool_call['id']

                print(f" -> [ACCIÓN DETECTADA] Invocar herramienta: '{nombre_tool}'")

                if isinstance(argumentos, dict) and argumentos:
                    parametro = list(argumentos.values())[0]
                else:
                    parametro = str(argumentos)

                print(f" -> [INPUT ENVIADO]: {parametro}")

                if nombre_tool in dict_herramientas:
                    resultado_obs = dict_herramientas[nombre_tool](parametro)
                else:
                    resultado_obs = f"Error: La herramienta '{nombre_tool}' no existe."

                print(f" -> [OBSERVACIÓN RECIBIDA]: {resultado_obs}")
                mensajes.append(ToolMessage(content=str(resultado_obs), tool_call_id=id_tool))
        else:
            print("\n==================================================================")
            print("                    RESPUESTA FINAL DEL AGENTE")
            print("==================================================================")
            print(respuesta.content)
            print("==================================================================")
            break

if 'llm_configurado' in locals() and llm_configurado is not None:
    if 'pipeline_rag' in locals():
        pregunta_analitica = "Dame un desglose completo, fila por fila, de absolutamente todos los registros de ventas asociados a Spain, sin importar la línea de producto?"
        ejecutar_agente_nativo_mistral(pregunta_analitica, llm_configurado, herramientas_agente)
    else:
        print("[ERROR] Por favor, ejecuta primero la celda de la Fase 7 para inicializar 'pipeline_rag'.")
else:
    print("[ERROR CRÍTICO] Se requiere la inicialización previa de Mistral Large 3 en la Fase 6.")

[SISTEMA] Inicializando Agente Autónomo con Function Calling Nativo...

[PENSAMIENTO] El agente analiza el contexto actual (Paso 1)...
 -> [ACCIÓN DETECTADA] Invocar herramienta: 'Buscador_Ventas_RAG'
 -> [INPUT ENVIADO]: todos los registros de ventas asociados a Spain, sin importar la línea de producto
 -> [OBSERVACIÓN RECIBIDA]: Basándome estrictamente en el contexto provisto, los registros de ventas asociados a **Spain** son los siguientes:

1. **Pedido Nro: 10424** (Fecha: 2005-05-31, Estado: IN PROCESS, Línea de Producto: TRUCKS AND BUSES)
   - 3 registros con cantidades y montos distintos:
     - 44 unidades → 2702.04 USD
     - 54 unidades → 7182.0 USD
     - 49 unidades → 7969.36 USD
   - Cliente: Euro Shopping Channel.

2. **Pedido Nro: 10246** (Fecha: 2004-05-05, Estado: SHIPPED, Línea de Producto: TRUCKS AND BUSES)
   - 35 unidades → 1704.5 USD
   - Cliente: Euro Shopping Channel.

**Nota:** Todos los registros disponibles en el corpus corresponden a la línea de producto *TR